# 01 Dataset Inventory

This notebook inventories the raw datasets available in the project workspace.

It will:
- scan `data/raw/datasets/`
- list files and subfolders
- preview tabular dataset files
- record basic metadata like rows, columns, and file sizes
- export summary CSVs to `outputs/`


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 200)


In [ ]:
# Project paths
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATASETS_DIR = PROJECT_ROOT / 'data' / 'raw' / 'datasets'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('DATASETS_DIR =', DATASETS_DIR)
print('OUTPUT_DIR    =', OUTPUT_DIR)

assert DATASETS_DIR.exists(), f'Datasets directory not found: {DATASETS_DIR}'


## 1. Show dataset folders


In [ ]:
dataset_dirs = sorted([p for p in DATASETS_DIR.iterdir() if p.is_dir()])
dataset_names = [p.name for p in dataset_dirs]

display(pd.DataFrame({'dataset_folder': dataset_names}))


## 2. File tree summary


In [ ]:
records = []

for dataset_dir in dataset_dirs:
    for path in sorted(dataset_dir.rglob('*')):
        if path.is_file():
            records.append({
                'dataset_folder': dataset_dir.name,
                'relative_path': str(path.relative_to(DATASETS_DIR)),
                'filename': path.name,
                'suffix': path.suffix.lower(),
                'size_kb': round(path.stat().st_size / 1024, 2),
            })

files_df = pd.DataFrame(records)
files_df = files_df.sort_values(['dataset_folder', 'relative_path']).reset_index(drop=True)
display(files_df)


## 3. Find tabular dataset files


In [ ]:
TABULAR_EXTS = {'.csv', '.xlsx', '.xls', '.tsv'}

tabular_df = files_df[files_df['suffix'].isin(TABULAR_EXTS)].copy()
tabular_df = tabular_df.reset_index(drop=True)
display(tabular_df)


## 4. Read each dataset file and collect metadata


In [ ]:
def load_table(path: Path):
    suffix = path.suffix.lower()
    if suffix == '.csv':
        return pd.read_csv(path)
    elif suffix == '.tsv':
        return pd.read_csv(path, sep='\t')
    elif suffix in {'.xlsx', '.xls'}:
        return pd.read_excel(path)
    else:
        raise ValueError(f'Unsupported file type: {suffix}')

inventory_rows = []
preview_tables = {}

for _, row in tabular_df.iterrows():
    file_path = DATASETS_DIR / row['relative_path']
    try:
        df = load_table(file_path)
        preview_tables[row['relative_path']] = df.head(5)
        inventory_rows.append({
            'dataset_folder': row['dataset_folder'],
            'relative_path': row['relative_path'],
            'filename': row['filename'],
            'rows': len(df),
            'cols': len(df.columns),
            'columns': ' | '.join(map(str, df.columns.tolist())),
            'missing_values_total': int(df.isna().sum().sum()),
            'duplicate_rows': int(df.duplicated().sum()),
        })
    except Exception as e:
        inventory_rows.append({
            'dataset_folder': row['dataset_folder'],
            'relative_path': row['relative_path'],
            'filename': row['filename'],
            'rows': None,
            'cols': None,
            'columns': f'ERROR: {e}',
            'missing_values_total': None,
            'duplicate_rows': None,
        })

inventory_df = pd.DataFrame(inventory_rows)
inventory_df = inventory_df.sort_values(['dataset_folder', 'relative_path']).reset_index(drop=True)
display(inventory_df)


## 5. Preview first rows of each readable dataset


In [ ]:
for rel_path, preview_df in preview_tables.items():
    display(Markdown(f'### `{rel_path}`'))
    display(preview_df)


## 6. High-level summary by dataset folder


In [ ]:
summary_df = (
    inventory_df.groupby('dataset_folder', dropna=False)
    .agg(
        file_count=('filename', 'count'),
        total_rows=('rows', 'sum'),
        total_missing=('missing_values_total', 'sum'),
        total_duplicates=('duplicate_rows', 'sum')
    )
    .reset_index()
)

display(summary_df)


## 7. Save outputs


In [ ]:
inventory_csv = OUTPUT_DIR / 'dataset_inventory_summary.csv'
files_csv = OUTPUT_DIR / 'dataset_file_tree.csv'
summary_csv = OUTPUT_DIR / 'dataset_folder_summary.csv'

inventory_df.to_csv(inventory_csv, index=False, encoding='utf-8-sig')
files_df.to_csv(files_csv, index=False, encoding='utf-8-sig')
summary_df.to_csv(summary_csv, index=False, encoding='utf-8-sig')

print('Saved:', inventory_csv)
print('Saved:', files_csv)
print('Saved:', summary_csv)


## 8. Manual next-step notes

After running this notebook, update:
- `data/raw/dataset_notes/dataset_inventory.md`
- `experiment_log.md`

Then send the notebook outputs back for the next step.
